# 해싱 기법 알아보기

해시는 특정 객체를 **정수값**으로 바꾸는 과정입니다.

In [2]:
print(hash("hello"))
print(hash(10))
print(hash((1, 2, 3)))

-1078400858779571561
10
529344067295497451


위의 값은 같은 방식의 해시화는 항상 같은 값이 나오게 됩니다.

**하지만** 플그램을 껐다 키면 문자열 hash값은 달라질 수 있습니다. 이는 Python이 보안을 위해 `str`, `bytes`같은 타입에 **hash randomization**을 적용하기 때문입니다.

# 해시화방식
**해시화**는 특정 문자열/숫자 등을 문자열의 문자들, 길이, 내부 바이트 정보등을 섞어서 가능한 고르게 퍼지는 정수값을 만드는 과정입니다.

이때, 임의의 해시값 `123456789012345`이 만들어 졌을때, `set` 내부 테이블이 8칸이면 실제로 저장 가능한 인덱스는 `0`~`7` 까지밖에 존재하지 않습니다.

이를 위해서는 해시값을 `0`~`7`로 줄여야하는 문제가 발생합니다.

때문에 `123456789012345 % 8 = 1` 등과 같은 연산을 통해 인덱스를 추립니다.

Python은 `& 7`과 같은 비트 연산자를 사용합니다.

> 타입마다 `__hash__()`에 의해 해시를 만드는 방법이 다릅니다.

## 해싱의 충돌 대응 방식

하지만 무한한 해시 값을 8개로 줄이게 될 경우, 겹치는 영역이 발생할 수 있습니다.

예를 들어서 `123456789012345`와 `123456789012353`은 저장 인덱스가 `1`로 같습니다.

이를 해결하기 위한 방법은 크게 2가지가 존재합니다.

### 1. 충돌 회피
애초에 충돌이 덜 나게 설계하는 방식입니다.
#### a. 좋은 해시 함수 사용
좋은 해시함수는 값을 잘 섞어서 여러 칸에 고르게 퍼트리게 됩니다. 간단한 시기이 아닌 다른 값이면 대부분 다른 해시값이 나오게, 유사할 수록 비슷한 위치에, 빠른 계산, 공격자가 충돌 발생시키기 어렵게 만들어집니다.

#### b. 테이블 크기 관리
해시테이블은 보통 내부 배열을 가지고 있습니다.

이때, 칸이 너무 적으면 충돌이 많아지기 때문에 데이터가 많아지면 2비트 형태로 내부 테이블을 키우게 됩니다. 이를 **rersizing**라고 합니다.

#### c. 로드 팩터 관리
로드팩터는 **테이블이 얼마나 찼는지**입니다.

이때, 로드팩터가 높아질수록, 빈칸을 찾기가 어려워지고 충돌이 일어날 수 있습니다. 이 때문에 꽉 차는 경우가 아니라 일정 비율 이상 차면 테이블을 키우게 됩니다. 이를 통해 비용은 들어도 전체적으로 검색/삽입 속도ㅗ를 빠르게 유지할 수 있습니다.


### 2. 충돌 처리
충돌이 났을 때 해결이 되는 저장방식입니다.
#### a. 체이닝
set의 저장 공간 배열에 같은 index가 여러 값이 오면 그 칸에 줄줄이 연결해서 저장하는 방식입니다. 예를 들어서
`set[1]: 123456789012345 > 123456789012353` 와 같은 형태로 저장하는 방식입니다.

이렇게 하면 데이터를 저장하기 쉬워지고, 해시 테이블의 크기를 키울 필요가 없으며 검색할 때 값을 찾을 때까지 체인을 타고 검색하는 시간이 소요되긴 하지만 계산 자체는 어렵지 않게 접근이 가능합니다.

> 참고로 이는 **Java**의 `HashMap`의 구현 방식입니다. 충도로이 너무 많아지면 트리 구조로 바뀌기도 합니다.

#### b. 오픈 어드레싱
이는 Python의 `set` 구현 방식에 가깝습니다.

이는 충돌이 난다면 다른 후보 `index`를 계산해서 빈칸을 찾을때까지 이동하는 방식입니다.

오픈 어드레싱의 방식은
- 선형 탐사: 3번에서 충돌나면 4, 5, 6, ... 순서로 빈칸이 보일때까지 찾기. 단순하고 캐시 효율이 좋지만 값들이 연속 구간에 뭉치기 쉬워 탐색이 느려질 수 있습니다.
- 제곱탐사: 충돌이 나면 1, 4, 9, ... 같이 점점 크게 건너뛰게 됩니다. 이덕분에 선형 탐사보다 클러스트링 문제가 덜 발생합니다. 이는 테이블 크기와 탐사 공식 설계가 중요합니다.

#### c. 이중 해싱
해시 함수를 2개 쓰는 방식입니다.

- 1번 해시: 시작 위치
- 2번 해시: 이동 간격

형태로 해싱을 시켜서 값들이 덜 뭉치고, 분포가 좋게 설계가 가능합니다.

하지만 해시 계산비용과 구현이 복잡합니다.

#### d. Python식 perturb 탐사
충돌 시 해시값을 흔들어서 더 넓게 퍼지게 탐색합니다. **오픈 어드레싱**과 함께 쓰입니다.

### 이외
외에도
- Robin Hood Hashing: 멀리 밀려난 값에 우선권 주기.
- Cuckoo Hashing: 두개 이상의 위치 후보를 갖는 방식.
- Hopscotch Hashing: 값을 원래 근처에 최대한 두는 방식.
- Perfect Hashing: 충돌이 아예 없도록 해시함수 만들기. (저장할 값이 정해져있을 때 사용 가능)

In [4]:
print(123456789012345 % 8)

1


In [24]:
s = {1, 2, 3, 4, 4}
print(s)

s.add("asdf")
print(s)

print("s.update([1, 'hello', [123, 123]]) 테스트")
try: 
  s.update([1, "hello", [123, 123]])
  print(s)
except TypeError as e:
  print(e, "발생")
  print("hashable 객체만 넣을 수 있습니다.")
  print("hashable이란? __hash__()가 있으며 해시값이 객체 생명동안 변하지 않는 값")

s.update([1, "hello"])
print(s)

s.remove(1)

print("try: s.remove('123asdfasg')")
try: 
  s.remove('123asdfasg')
except KeyError as e:
  print("KeyError:",e)


print("try: print(s.discard('123asdfasg'))")
try: 
  print(s.discard('123asdfasg'))
  print("discard는 없으면 None 반환")
except KeyError as e:
  print("KeyError:",e)

{1, 2, 3, 4}
{1, 2, 3, 4, 'asdf'}
s.update([1, 'hello', [123, 123]]) 테스트
unhashable type: 'list' 발생
hashable 객체만 넣을 수 있습니다.
hashable이란? __hash__()가 있으며 해시값이 객체 생명동안 변하지 않는 값
{1, 2, 3, 4, 'asdf', 'hello'}
try: s.remove('123asdfasg')
KeyError: '123asdfasg'
try: print(s.discard('123asdfasg'))
None
discard는 없으면 None 반환
